# p4p basics: simulated motors and camera

This notebook shows how to connect with p4p, read simulated motor and camera PVs, move the motors, and inspect simple camera values.

In [ ]:
import time
from p4p.client.thread import Context

ctx = Context("pva")
MOTX = "LAB:SIM:HMIR"
MOTY = "LAB:SIM:VMIR"
CAM = "LAB:SIM:CAM01"

def unwrap_scalar(value):
    if hasattr(value, "value"):
        value = value.value
    if isinstance(value, dict) and "value" in value:
        value = value["value"]
    return value

def get_scalar(pv):
    return unwrap_scalar(ctx.get(pv, timeout=2.0))

def put_scalar(pv, value, wait=True):
    ctx.put(pv, value, wait=wait, timeout=5.0)

def wait_motor_done(motor_pv, timeout=10.0):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if float(get_scalar(f"{motor_pv}.DMOV")) == 1.0:
            return True
        time.sleep(0.1)
    return False

In [ ]:
state = {
    "HMIR_RBV": float(get_scalar(f"{MOTX}.RBV")),
    "VMIR_RBV": float(get_scalar(f"{MOTY}.RBV")),
    "CentroidX": float(get_scalar(f"{CAM}:Stats1:CentroidX_RBV")),
    "CentroidY": float(get_scalar(f"{CAM}:Stats1:CentroidY_RBV")),
    "PeakStartX": float(get_scalar(f"{CAM}:PeakStartX")),
    "PeakStartY": float(get_scalar(f"{CAM}:PeakStartY")),
}
state

In [ ]:
# Move the simulated motors and wait for completion.
put_scalar(f"{MOTX}.VAL", 5.0, wait=False)
put_scalar(f"{MOTY}.VAL", -2.0, wait=False)
wait_motor_done(MOTX)
wait_motor_done(MOTY)

{
    "HMIR_RBV": float(get_scalar(f"{MOTX}.RBV")),
    "VMIR_RBV": float(get_scalar(f"{MOTY}.RBV")),
    "CentroidX": float(get_scalar(f"{CAM}:Stats1:CentroidX_RBV")),
    "CentroidY": float(get_scalar(f"{CAM}:Stats1:CentroidY_RBV")),
}

In [ ]:
# Write simple camera simulation parameters and read them back.
put_scalar(f"{CAM}:PeakStartX", 320)
put_scalar(f"{CAM}:PeakStartY", 260)
time.sleep(0.5)

{
    "PeakStartX": float(get_scalar(f"{CAM}:PeakStartX")),
    "PeakStartY": float(get_scalar(f"{CAM}:PeakStartY")),
    "CentroidX": float(get_scalar(f"{CAM}:Stats1:CentroidX_RBV")),
    "CentroidY": float(get_scalar(f"{CAM}:Stats1:CentroidY_RBV")),
}